# Match metadata: ALLSHEETS residual pass

This notebook runs the residual ALLSHEETS path after a DATA metadata assignment exists.
It builds residual vectors, runs ALLSHEETS matching, assigns metadata, and combines sessions.


In [3]:
# Shared roots and latest DATA metadata session for residual matching.
from src.rainfall_rescue_sqlite.parquet_ingest import default_ensemble_parquet_root, default_rainfall_rescue_parquet_root
from src.rainfall_rescue_sqlite.parquet_similarity import default_comparison_parquet_root

rr_dataset_root = default_rainfall_rescue_parquet_root()
ensemble_dataset_root = default_ensemble_parquet_root()
comparison_root = default_comparison_parquet_root()

meta_dir = comparison_root / "ensemble_metadata"
meta_files = sorted(meta_dir.glob("session_*.parquet"), key=lambda p: int(p.stem.split("_")[1]))
if not meta_files:
    raise FileNotFoundError("No DATA metadata session found. Run the active runbook first.")
data_metadata_path = meta_files[-1]
print(f"Using DATA metadata session: {data_metadata_path}")


Using DATA metadata session: /Volumes/Scratch/ADRQ/monthly_similarity_parquet/ensemble_metadata/session_000003.parquet


## Match residual records against ALLSHEETS

The assignment above matches ensemble records against the combined **DATA**
Rainfall-Rescue records. Many transcriptions have no exact match there, either
because the station-year is not in `DATA` or because it was transcribed only on
an individual **ALLSHEETS** source sheet.

As an **additional step**, every ensemble record *without* an exact DATA match
(i.e. `match_type` is `approximate` or unmatched) is re-run through the **same
matching algorithm** against the ALLSHEETS dataset:

1. Build ALLSHEETS candidate vectors and reuse the residual ensemble query
   vectors in a separate comparison root
   (`monthly_similarity_allsheets_parquet`), leaving the DATA pass untouched.
2. Run the identical baseline matcher against the ALLSHEETS candidates.
3. Assign metadata with the identical exact-match rule.
4. **Combine** the two passes into a new, final `ensemble_metadata` session:
   DATA exact matches are kept as-is, and residual records are filled from
   ALLSHEETS.

ALLSHEETS source sheets carry **no coordinates**, so an ALLSHEETS match supplies
`matched_location_name` and `matched_year` directly and is tagged
`match_type = 'exact_allsheets'`. During the combine step the location name is
looked up (case- and punctuation-insensitively) in `LeftOverSites.csv`; where a
single unambiguous coordinate is found the `matched_latitude`,
`matched_longitude` and `matched_elevation_ft` are filled in. Coordinate-filled
`exact_allsheets` records are then SEF-exportable alongside the DATA exact
matches, while any that stay coordinate-less remain available for name-based
diagnostics only. The combined session becomes the latest one, so downstream
code picks it up automatically.


In [4]:
# Set up the ALLSHEETS residual-matching paths.
#
# `data_metadata_path` is expected from Cell 2 in this notebook. If Cell 2 has
# not been run in this kernel, fall back to the latest DATA metadata session.

from pathlib import Path

from src.rainfall_rescue_sqlite.parquet_ingest import default_allsheets_parquet_root
from src.rainfall_rescue_sqlite.parquet_similarity import (
    default_allsheets_comparison_parquet_root,
    default_comparison_parquet_root,
 )

allsheets_dataset_root = default_allsheets_parquet_root()
allsheets_comparison_root = default_allsheets_comparison_parquet_root()

if "data_metadata_path" in globals() and data_metadata_path is not None:
    data_metadata_path = Path(data_metadata_path)
else:
    comparison_root = default_comparison_parquet_root()
    meta_dir = comparison_root / "ensemble_metadata"
    meta_files = sorted(
        meta_dir.glob("session_*.parquet"),
        key=lambda p: int(p.stem.split("_")[1]),
    )
    if not meta_files:
        raise FileNotFoundError(
            "No DATA metadata session found. Run the runbook metadata assignment first."
        )
    data_metadata_path = meta_files[-1]

print(f"Using DATA metadata session: {data_metadata_path}")
allsheets_dataset_root, allsheets_comparison_root, data_metadata_path


Using DATA metadata session: /Volumes/Scratch/ADRQ/monthly_similarity_parquet/ensemble_metadata/session_000003.parquet


(PosixPath('/Volumes/Scratch/ADRQ/Rainfall-Rescue/rainfall_rescue_allsheets_parquet'),
 PosixPath('/Volumes/Scratch/ADRQ/monthly_similarity_allsheets_parquet'),
 PosixPath('/Volumes/Scratch/ADRQ/monthly_similarity_parquet/ensemble_metadata/session_000003.parquet'))

In [5]:
# Build the ALLSHEETS comparison vectors for the residual ensemble records.
#
# - Candidate side (`rr_monthly_vectors`) is rebuilt from the ALLSHEETS dataset.
# - Query side reuses the DATA comparison root, restricted to residual files
#   (those with `match_type != 'exact'` in the DATA assignment).

from src.rainfall_rescue_sqlite.parquet_similarity import (
    build_allsheets_comparison_vectors_parquet,
)

allsheets_build = build_allsheets_comparison_vectors_parquet(
    allsheets_dataset_root=allsheets_dataset_root,
    source_comparison_root=comparison_root,
    allsheets_comparison_root=allsheets_comparison_root,
    data_metadata_path=data_metadata_path,
    overwrite=True,
)
print(
    f"ALLSHEETS candidate vectors: {allsheets_build.rr_vectors:,}\n"
    f"Residual ensemble query vectors: {allsheets_build.ensemble_vectors:,}"
)


ALLSHEETS candidate vectors: 449,293
Residual ensemble query vectors: 62,205


In [7]:
# Run the baseline matcher against the ALLSHEETS candidates.
#
# Same algorithm/parameters as the DATA pass. As in that pass, this is bounded
# for an interactive demo; drop `max_ensemble_queries` for a full-scale run.

from src.rainfall_rescue_sqlite.parquet_similarity import run_baseline_matching_parquet

allsheets_session_summary = run_baseline_matching_parquet(
    comparison_root=allsheets_comparison_root,
    top_k=10,
    min_overlap=10,
    uncertainty_weight=0.15,
    max_ensemble_queries=200,
    max_rr_candidates=20000,
)
allsheets_session_summary


MatchResult(comparison_root=PosixPath('/Volumes/Scratch/ADRQ/monthly_similarity_allsheets_parquet'), session_id=1, ensemble_queries=200, rr_candidates=20000, matches_written=1950)

In [8]:
# Assign ALLSHEETS metadata for the residual records (same exact-match rule).

from src.rainfall_rescue_sqlite.assign_ensemble_metadata import (
    assign_ensemble_metadata_parquet,
 )

allsheets_result = assign_ensemble_metadata_parquet(
    comparison_root=allsheets_comparison_root,
    ensemble_dataset_root=ensemble_dataset_root,
    rr_dataset_root=allsheets_dataset_root,
    session_id=None,
)
print(
    f"ALLSHEETS exact matches: {allsheets_result.exact_matches:,}\n"
    f"ALLSHEETS approximate: {allsheets_result.approximate_matches:,} "
    "(never coordinate-resolved; ALLSHEETS has no coords)\n"
    f"Metadata written to: {allsheets_result.output_path}"
)


ALLSHEETS exact matches: 4
ALLSHEETS approximate: 0 (never coordinate-resolved; ALLSHEETS has no coords)
Metadata written to: /Volumes/Scratch/ADRQ/monthly_similarity_allsheets_parquet/ensemble_metadata/session_000001.parquet


In [9]:
# Combine the DATA and ALLSHEETS passes into the final ensemble_metadata session.
#
# DATA exact matches are kept; residual records are filled from ALLSHEETS exact
# matches (tagged 'exact_allsheets'); remaining DATA approximate matches are
# retained. ALLSHEETS sheets carry no coordinates, so where the location name is
# found in LeftOverSites.csv the lat/lon/elevation are filled in (making those
# records SEF-exportable). The combined session is written under the main DATA
# comparison root, becoming the latest one downstream code reads.

from src.rainfall_rescue_sqlite.assign_ensemble_metadata import (
    combine_metadata_assignments_parquet,
)

if "data_metadata_path" not in globals() or data_metadata_path is None:
    raise NameError(
        "data_metadata_path is not set. Run Cell 2 (setup) and Cell 4 first."
    )
if "allsheets_result" not in globals() or allsheets_result is None:
    raise NameError(
        "allsheets_result is not set. Run Cell 7 before this cell."
    )

combined = combine_metadata_assignments_parquet(
    data_metadata_path=data_metadata_path,
    allsheets_metadata_path=allsheets_result.output_path,
    comparison_root=comparison_root,
    # leftover_sites_csv defaults to LeftOverSites.csv under the RR root.
)
print(
    f"Combined session {combined.session_id} -> {combined.output_path}\n"
    f"  total ensemble files : {combined.total_ensemble_files:,}\n"
    f"  DATA exact           : {combined.data_exact:,}\n"
    f"  ALLSHEETS filled     : {combined.allsheets_filled:,}\n"
    f"  ALLSHEETS w/ coords  : {combined.allsheets_with_coords:,}\n"
    f"  DATA approximate     : {combined.data_approximate:,}\n"
    f"  unmatched            : {combined.unmatched:,}"
)


Combined session 4 -> /Volumes/Scratch/ADRQ/monthly_similarity_parquet/ensemble_metadata/session_000004.parquet
  total ensemble files : 62,216
  DATA exact           : 11
  ALLSHEETS filled     : 4
  ALLSHEETS w/ coords  : 0
  DATA approximate     : 0
  unmatched            : 62,201


In [10]:
# Inspect a sample of the ALLSHEETS-filled records in the combined session.
# Those whose name was found in LeftOverSites.csv now carry coordinates; the
# rest have a location name + year but no coordinates.

import duckdb

_con = duckdb.connect()
_sample = _con.execute(
    f"""
    SELECT file_id, file_name, matched_location_name, matched_year,
           matched_latitude, matched_longitude, match_type, match_source_session_id
    FROM read_parquet('{combined.output_path}')
    WHERE match_type = 'exact_allsheets'
    ORDER BY matched_latitude IS NULL, file_id
    LIMIT 15
    """
).df()
_con.close()
_sample


,file_id,file_name,matched_location_name,matched_year,matched_latitude,matched_longitude,match_type,match_source_session_id
0,1002,DRain_1861-1870_Durham-1.json,DINSDALE RECTORY,1865,NaN,NaN,exact_allsheets,4
1,10002,DRain_1861-1870_Essex-12.json,SOUTHCHURCH NR SOUTHEND,1861,NaN,NaN,exact_allsheets,4
2,10005,DRain_1861-1870_Worcestershire-5.json,ORLETON,1867,NaN,NaN,exact_allsheets,4
3,10115,DRain_1881-1890_Durham-178.json,LONDON,1839,NaN,NaN,exact_allsheets,4


### Full-scale run note

The cells above are intended for bounded, interactive checking.

For a full-scale local parallel run, use the operational workflow in
`match_metadata_operations.ipynb`.